# 08-芯片制造、CPU进阶与软硬件生态

> 从已经搭好的教学计算机出发，理解真实芯片如何制造、现代 CPU 如何提高性能、程序如何跨越软硬件边界，以及这些环节为什么都涉及安全与信任。

本篇是《自己动手做台计算机》**8 篇笔记的最后一篇**，覆盖课程末尾的后门和漏洞、光刻工艺、流水线、高级 CPU 技术、软硬件生态系统与编程语言发展六个主题。

## 1. 学习范围与资料核对说明

| 课程视频 | 目录标题 | 本篇学习目标 |
|---|---|---|
| 76 | 后门和漏洞 | 区分缺陷与隐藏通道，认识软硬件信任边界 |
| 77 | 光刻工艺 | 理解电路设计如何变成硅片上的器件和连线 |
| 78 | 流水线技术 | 用周期表理解多条指令重叠执行 |
| 79 | 高级CPU技术 | 区分缓存、超标量、乱序、推测与多核 |
| 80 | 软硬件生态系统 | 串起指令集、工具链、操作系统和应用 |
| 81 | 编程语言发展 | 理解机器码、汇编、高级语言与执行方式 |

**资料核对边界：**已核对上述视频的目录标题。本篇没有逐段观看或转录这六个视频，也未核验其中具体举例、参数和实现。下文的详细解释、示意图、公式和练习均作为**补充理解**编写，用于建立可复习的知识框架，不作为讲师原话或课程电路的新增设计。

配套讲义为 101 页的《自己动手做台计算机.pdf》。第 07 篇已核对的讲义末尾主要是教学 CPU 的程序、微操作和控制表，不能据此给本篇六个主题编造对应页码。本篇不附未经核验的课件截图。

## 2. 从教学 CPU 走向真实计算机

前七篇依次建立了数据表示、逻辑门、ALU、寄存器和内存、CPU 数据通路、指令与程序、自动控制和完整系统。第 07 篇中的教学 CPU 已经能够在控制单元安排下执行多步取指与操作。

本篇换一个观察尺度：同样的逻辑功能，怎样做成芯片、怎样更快地执行更多程序、怎样与工具和软件协同？

| 前面学过的内容 | 本篇延伸的问题 |
|---|---|
| 逻辑门与 PCB 接线 | 大量逻辑门怎样集成到芯片中？ |
| 寄存器、ALU 和控制单元 | 怎样让多条指令同时处于不同执行阶段？ |
| CPU 与内存之间的数据传送 | 主存较慢时，怎样减少等待？ |
| 指令集和汇编 | 不同工具怎样约定指令、寄存器和调用方式？ |
| CPU、内存与输入输出 | 软件怎样取得访问设备的权限？ |
| 自动执行程序 | 出错或被恶意修改的程序会产生什么影响？ |

**补充理解：**后面使用的五级流水线、通用寄存器指令和缓存示意都是解释模型。前面搭建的教学 CPU 采用多个控制步骤执行指令，不能直接认定它已经具备这些现代处理器功能。

## 3. 后门与漏洞：错误行为从哪里来？

以下为对视频 76 主题的**补充理解**。

### 3.1 三个容易混淆的概念

| 概念 | 含义 | 示例 |
|---|---|---|
| 普通缺陷 | 实现不符合预期，但未必构成安全问题 | 某个显示值计算错误 |
| 安全漏洞 | 可被利用来破坏保密性、完整性或可用性的弱点 | 没有检查边界的写入造成越界，进一步影响程序执行 |
| 后门 | 有意留下、绕过正常安全控制的访问或控制机制 | 隐藏口令绕过公开的身份验证 |

漏洞可能来自无意的设计或实现错误；后门强调有意设置的绕过机制。仅仅发现异常行为，不能据此判断开发者的动机。

远程维护、调试接口和固件更新功能本身不自动等于后门。应检查它们是否公开、是否有认证和授权、是否受到预期的安全控制。长期保留的默认凭据也可能形成严重风险，即使不能证明存在恶意意图。

### 3.2 软件和硬件都可能形成弱点

- 应用程序：输入检查、访问权限和数据处理出错。
- 操作系统与驱动：更高权限的代码存在缺陷，影响范围可能扩大。
- 固件：启动过程、更新机制或设备控制逻辑存在问题。
- 硬件：访问隔离、调试控制或器件实现存在弱点。
- 系统配置：使用正确的组件，但组合、权限或部署方式不正确。

例如，机器人本应只接受授权控制端的电机指令。如果控制接口没有认证，攻击者可能利用这一漏洞发送指令；如果设备中另有一组隐藏口令可以绕过认证，则涉及后门机制。

## 4. 信任链与嵌入式设备安全

以下为**补充理解**，用于把安全问题放回计算机系统中。

```text
上电 → 启动代码 / Boot ROM → 引导程序与固件
  → 操作系统或运行时 → 应用程序 → 传感器、通信接口和执行器
```

每一层都依赖下面的组件按约定工作。安全设计要说明：**谁有权修改代码，谁有权读取数据，谁有权控制设备，系统如何确认这些权限？**

| 环节 | 常见保护措施 | 解决的主要问题 |
|---|---|---|
| 启动与更新 | 校验固件签名、保护信任根、按需求防止回滚 | 减少未经授权的代码替换 |
| 通信与控制 | 身份认证、授权、完整性保护 | 限制谁能发出有效控制指令 |
| 程序与内存 | 输入检查、权限隔离、内存保护 | 限制错误或攻击造成的影响 |
| 调试与生产配置 | 按部署需求管理调试口和凭据 | 减少额外访问入口 |
| 物理执行 | 独立急停、限位、看门狗和失效处理 | 软件异常时限制物理后果 |

**两个边界必须记住：**

1. 校验签名证明的是“代码由被信任的密钥授权”，不能证明代码没有漏洞；若要求防止旧版本回滚，还需要额外机制。
2. 网络安全与控制安全需要协同。控制接口经过认证，也不能代替机械限位或急停；看门狗用于恢复失去响应的系统，也不能独自识别全部错误控制动作。

## 5. 芯片制造：把逻辑变成物理结构

以下为对视频 77 主题的**补充理解**。

前面画出的逻辑门是功能层面的表达。集成电路制造还要决定晶体管、接触孔和金属连线在芯片上的位置，并通过多道工序形成这些结构。

```text
设计侧：功能需求 → 电路/逻辑设计 → 验证 → 布局布线 → 版图与制造数据
制造侧：晶圆准备 → 多轮图形化与材料加工 → 晶圆测试 → 切割 → 封装 → 成品测试
```

### 晶圆、裸片和封装

- **晶圆（Wafer）**：用于批量制造器件的圆形基片，常见材料是硅。
- **裸片（Die）**：晶圆上的一个独立电路单元，切割后得到。
- **封装（Package）**：为裸片提供电气连接、机械保护和散热路径等。

开发板上焊接的“芯片”通常是已经封装的器件；它不是一块只经过一次照射的硅片。一个封装也可能包含多个裸片。芯片内的器件和互连形成逻辑功能；PCB 则在板级连接封装器件、电源和外设。

## 6. 光刻完成什么：形成加工用的图形

以下为**补充理解**，用常见光学光刻说明基本思想，不代表课程采用的某一种具体设备或工艺。

### 6.1 一次图形化过程的简化步骤

1. 准备表面，在本轮要加工的材料层上形成合适条件。
2. 涂覆光刻胶，形成对曝光敏感的薄膜。
3. 对准与曝光，把需要的图形映射到光刻胶上。
4. 显影，利用曝光后的溶解性差异得到光刻胶图形。
5. 结合刻蚀、注入或其他工艺，让选定区域发生加工。
6. 去除不再需要的掩膜并检查，为后续工序准备表面。

```text
设计图形 → 对准、曝光、显影 → 光刻胶图形 → 与其他加工步骤配合 → 目标结构
```

**光刻主要解决“在哪些位置加工”的问题。**沉积形成材料，刻蚀去掉材料，掺杂等过程改变器件区域的性质，清洗、热处理和平坦化也可能参与制造。不同工序的先后顺序依工艺而定。

### 6.2 为什么必须反复加工？

晶体管结构、接触孔以及多层金属互连需要多次图形化和加工。每一轮都要与前面的结构正确对准。一次曝光不会同时生成整颗 CPU 的全部晶体管和所有连线。

### 6.3 怎样理解工艺节点？

“某某纳米工艺”是工艺代际的名称，不能直接理解为每个晶体管尺寸、每条导线宽度都恰好等于这个数值。不同厂商的同名节点也不能仅按名称比较。先进工艺可能改善密度、性能或能效，但产品表现还取决于电路设计、电压、散热、成本、良率和封装。

## 7. 逻辑正确，不等于芯片已经能工作

以下为**补充理解**。设计验证和制造测试有不同目标。

| 检查层次 | 主要检查什么 |
|---|---|
| 功能验证 | 指令、状态变化和输出是否符合设计要求 |
| 时序检查 | 信号是否能在规定时间内到达并被正确采样 |
| 物理检查 | 版图是否满足工艺规则，连接是否符合电路 |
| 电源与热设计 | 电压、供电和散热是否支持预期运行 |
| 制造与成品测试 | 实际器件是否有缺陷，是否达到要求 |

例如，ALU 在仿真中算出正确结果，只证明相关逻辑在测试条件下表现正确。真实实现中，较长的信号路径可能无法赶在时钟边沿前稳定；供电不足也可能导致异常。

这与第 07 篇的控制过程有关：一个时钟边沿能否可靠保存数据，除了 OE、WE 等控制关系正确，还受传播延迟、建立时间和保持时间约束。

**不要把三件事混为一谈：**设计按规格正确、工艺可以制造、制造出的器件经过测试合格。它们需要不同的检查。

## 8. 流水线：让多条指令在不同阶段工作

以下为对视频 78 主题的**补充理解**。采用经典的五级教学模型，实际 CPU 不一定按这五级划分。

| 阶段 | 简称 | 在本模型中的任务 |
|---|---|---|
| 取指 | IF | 读取指令 |
| 译码 / 读寄存器 | ID | 识别操作并读取源寄存器 |
| 执行 | EX | 完成 ALU 运算或计算访存地址 |
| 访存 | MEM | 对需要访存的指令访问数据存储器 |
| 写回 | WB | 对需要写回的指令更新目的寄存器 |

不采用流水线时，可以先让 I1 走完全部步骤，再处理 I2。采用流水线后，I1 进入下一阶段时，I2 可以进入第一阶段。**不同阶段之间需要保存中间结果和控制信息的流水线寄存器。**

```text
IF → [阶段寄存器] → ID → [阶段寄存器] → EX → [阶段寄存器] → MEM → [阶段寄存器] → WB
```

普通算术指令通常不需要在 MEM 阶段读数据，存储指令通常不需要在 WB 阶段写目的寄存器。流水线改变内部实现方式；只要保持指令集规定的结果，程序就不需要知道内部用了多少级。前面教学 CPU 的取指微操作不能与这里的阶段逐项等同。

## 9. 逐周期看流水线：四条指令需要多久？

以下为**补充例子**。假设每阶段用一个周期，指令之间没有依赖和资源冲突，不发生分支、缓存未命中或其他停顿。

| 指令 \ 周期 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| I1 | IF | ID | EX | MEM | WB | — | — | — |
| I2 | — | IF | ID | EX | MEM | WB | — | — |
| I3 | — | — | IF | ID | EX | MEM | WB | — |
| I4 | — | — | — | IF | ID | EX | MEM | WB |

从表中读出：

- 第一条指令经过 5 个周期才完成。
- 第 5—8 周期，每周期有一条指令完成。
- 四条指令共用 8 个周期。
- 若相同步骤完全不重叠执行，则需要 4 × 5 = 20 个等长阶段周期。

这里提高的是**连续处理多条指令的吞吐量**，不是把每条指令的五个步骤减少为一个步骤。

| 指标 | 问的是 | 本例 |
|---|---|---|
| 延迟 | 一条指令从开始到完成多久？ | 每条经过 5 个周期 |
| 吞吐量 | 一段时间能完成多少条？ | 充满后理想地每周期 1 条 |
| 整段完成时间 | 从第一条开始到最后一条完成多久？ | 4 条共 8 个周期 |

## 10. 流水线性能公式与成立条件

以下为**补充推导**。设流水线有 k 级、连续执行 N 条指令，每级耗时 t，并暂时忽略阶段寄存器开销和所有停顿。

```text
不重叠执行时间：T串行 = N × k × t
流水线执行时间：T流水 = (k + N - 1) × t
理想加速比：    S = Nk / (k + N - 1)
```

本篇的 k=5、N=4：T串行 = 20t，T流水 = 8t，S = 20 / 8 = 2.5。当 N 很大时，填充和排空的影响变小，理想加速比趋近于 k；N=1 时没有指令重叠收益。

流水线周期至少受最慢阶段和寄存器开销限制：

```text
流水线周期 ≥ 最慢阶段的组合逻辑延迟 + 阶段寄存器相关开销
```

若五个阶段的组合延迟分别为 1、1、3、1、1 ns，寄存器相关开销取 0.2 ns，周期至少约为 3.2 ns。按同一简化模型，单条指令延迟约为 16 ns，连续四条约为 8 × 3.2 = 25.6 ns。这些数值只用于说明比较方法，不能当作任何真实 CPU 的实测性能。增加级数也不保证一直变快。

## 11. 流水线冒险：什么时候不能顺利重叠？

以下为**补充理解**。“冒险（Hazard）”表示重叠执行遇到资源或正确性约束。

| 类型 | 原因 | 简化例子 | 常见处理方法 |
|---|---|---|---|
| 结构冒险 | 两个阶段同时需要容量不足的资源 | 取指和数据访问共用一个只能处理一次访问的端口 | 分离资源、增加端口、仲裁或停顿 |
| 数据冒险 | 后一条需要前一条尚未提供的结果 | ADD 生成 R1，紧接着 SUB 读取 R1 | 检测依赖、前递或停顿 |
| 控制冒险 | 尚不确定接下来应取哪条指令 | 条件分支的结果尚未算出 | 停顿、预测；错误时撤销错误路径 |

**停顿**是让某些阶段等待；**气泡**是不产生有效指令结果的空位置；**冲刷**通常用于丢弃已经进入流水线、但不应继续生效的工作。五级单发射模型在稳定、理想条件下可以接近 CPI=1，但不是无条件保证。

## 12. 数据依赖与分支推测

以下为**补充理解**。采用通用寄存器的假想汇编，目的寄存器写在第一位，**不是前面课程 CPU 的指令格式**。

```text
ADD R1, R2, R3     ; R1 ← R2 + R3
SUB R4, R1, R5     ; R4 ← R1 - R5
```

SUB 必须使用 ADD 生成的新 R1，这称为读后写依赖（RAW）。在典型五级流水线中，ADD 的结果在 EX 末尾形成，即使尚未完成 WB，也可能经旁路送给后一条 SUB 的 EX 输入。**前递没有消除依赖，只是让结果更早到达使用位置。**

加载后立即使用时，数据可能到 MEM 末尾才得到，而后续 ADD 原本更早就要在 EX 使用它，因此通常需要至少一个停顿；缓存未命中可能等待更久。具体停顿数取决于内存时序、流水线划分和旁路设计。

条件分支可能要等比较结果才能决定下一次取指地址，因此处理器可能预测方向并沿预测路径取指。结果确认后，预测错误就冲刷错误路径并恢复正确路径。错误路径不应提交为程序正式结果，但内部缓存状态等仍可能留下可观察差异，这说明性能机制也需要安全边界。

## 13. 高级 CPU 技术：缓存、超标量、乱序与多核

以下为对视频 79 主题的**补充理解**，是通用技术选讲，不表示视频逐项讲解了全部内容。

### 13.1 缓存和局部性

```text
寄存器 → L1 缓存 → L2 缓存 → 可能存在的共享末级缓存 → 主存
```

通常越往右容量越大、访问等待越长。**时间局部性**是刚用过的数据可能很快再次使用；**空间局部性**是用到某个地址后可能接着用附近地址。缓存未命中会带来额外等待。

一个简化模型是：`平均访问时间 = 命中时间 + 未命中率 × 未命中额外代价`。若命中时间 1 ns、未命中率 5%、额外代价 50 ns，平均约为 3.5 ns。真实系统还可能重叠多个访问，这不是整机执行时间公式。实时控制不能只看平均值。

### 13.2 几种技术不是一回事

| 技术 | 核心想法 |
|---|---|
| 超标量 | 每周期有能力发射多条指令到不同执行资源 |
| 乱序执行 | 让满足约束、已准备好的指令先工作 |
| 寄存器重命名 | 用不同内部位置减少名称复用造成的假依赖 |
| 推测执行 | 条件确认前按预测暂时执行，之后确认或撤销 |
| 多核 | 提供多个执行核心，安排不同线程或任务并行 |

超标量关注单个核心的发射宽度，乱序关注单个核心内部的执行安排，多核提供多个核心；三者可以组合，但不存在必然等同关系。

用 Amdahl 定律看多核收益：`S = 1 / [(1 - P) + P / n]`。若 75% 工作可并行、使用 4 核，理想加速比约 2.29，而不是 4。串行部分、锁竞争、内存带宽和任务分配都会继续限制收益。

### 13.3 乱序不意味着随意改变程序结果

假设有三条抽象操作：A 从较慢的内存取得 x，B 计算 y=x+1，C 计算 z=u+v。B 依赖 A，必须等待 x；如果 C 没有其他依赖，处理器可能先执行 C，提高资源利用率。它不能把 B 移到 A 前面并使用旧的 x。

真依赖要求使用前一条产生的值，重命名不能让尚未得到的值提前出现。名称复用造成的反依赖、输出依赖可以通过寄存器重命名减少。现代乱序处理器常通过按程序顺序提交等机制保持程序可见状态与异常行为正确。

**缓存一致性不等于线程安全。**即使不同核心对共享数据的观察满足一致性规则，读取—修改—写回这类多步操作仍可能需要原子操作或锁。

## 14. 性能不能只看主频，控制也不能只看平均速度

以下为**补充理解**。

```text
CPU 执行时间 = 执行的指令数 × 平均 CPI / 时钟频率
```

这是 CPU 执行时间关系式，程序总等待时间还可能包含设备、网络和调度。若执行 1 亿条指令、平均 CPI=2、频率为 1 GHz，CPU 时间为 0.2 s；若 CPI 降为 1.5，则为 0.15 s。但编译器、指令集和算法会改变指令数，不能单独比较一个指标。

| 使用场景 | 重点指标 |
|---|---|
| 大量图像处理 | 吞吐量、延迟、内存带宽、能效 |
| 电池设备 | 功耗、续航、待机与唤醒开销 |
| 固定周期电机控制 | 截止时间、最坏情况延迟与抖动 |
| 通信和交互 | 响应时间与整体系统等待 |

实时性关注规定工作是否能在截止时间前完成。“平均很快”不能保证每一次都及时。缓存未命中、分支误预测、中断和操作系统调度都可能增加波动。

## 15. 软硬件生态：程序与处理器怎样约定合作？

以下为对视频 80 主题的**补充理解**。

```text
应用需求与程序 → 语言、库、框架 → 编译器 / 汇编器 / 链接器 / 运行时
→ 操作系统与驱动（如果使用） → 指令集、调用约定、平台与设备接口
→ CPU、内存、总线、外设
```

| 层次或接口 | 约定什么 |
|---|---|
| ISA：指令集架构 | 指令含义、可见寄存器、编码以及执行规则 |
| 微架构 | 芯片内部怎样实现 ISA，如流水线、缓存和执行资源 |
| ABI：应用二进制接口 | 函数传参、返回值、寄存器使用和数据布局等 |
| 操作系统与系统接口 | 资源管理、权限、进程/线程以及系统服务 |
| 驱动与设备接口 | 设备寄存器、通信、数据传送及设备状态 |
| 工具链与库 | 怎样生成、链接、调试程序并提供复用功能 |

两个处理器支持相同 ISA，不代表任意二进制都能直接通用；还要检查扩展指令、ABI、操作系统、可执行文件格式和库。不同 ISA 也可能运行同一份高级语言源码，但通常需要重新编译或使用翻译环境。CPU 生态的价值不只在核心跑得快，也在工具、文档、库、驱动和应用能否协作。

## 16. 编程语言发展：把人的意图逐层变成执行动作

以下为对视频 81 主题的**补充理解**。

| 表达方式 | 主要特点 | 程序员需要处理的内容 |
|---|---|---|
| 机器码 | 处理器直接识别的指令编码 | 编码、寄存器、地址和机器约束 |
| 汇编语言 | 用助记符和符号表达机器指令 | 仍需理解目标 ISA 和执行模型 |
| 高级语言 | 用类型、表达式、函数等表达算法 | 关注逻辑，必要时理解资源与平台 |
| 库与框架 | 封装反复出现的复杂任务 | 理解接口、行为和限制 |

第 06 篇手工填写指令编码，是最接近机器码的一层。汇编器把助记符和标签转换为相应编码，减少人工算地址和填位的工作。高级语言又进一步表达循环、函数和数据结构。

一段高级语言程序的常见生成路径：

```text
源代码 → 编译器 → 汇编表示或目标代码 → 汇编 / 目标文件 → 链接 → 可执行文件或固件镜像 → 装载或烧录 → 处理器执行
```

提前编译（AOT）、解释执行、即时编译（JIT）和混合实现都是执行方式。同一语言在不同实现中可能采用不同方式，因此不宜简单说某语言永远“逐行解释”。裸机固件也可以不使用通用操作系统。

### 编译、链接和运行分别做什么？

| 环节 | 主要工作 |
|---|---|
| 编译 | 按语言规则把源码转换为中间表示或目标代码 |
| 汇编 | 把汇编表示转换为目标指令编码 |
| 链接 | 组合代码和数据，处理符号引用与地址关系 |
| 装载或烧录 | 把程序放到能够运行的位置 |
| 启动与运行时初始化 | 按需要准备栈、数据区域和库所需状态 |

工具链不一定生成单独可见的汇编文本，有些步骤在工具内部完成。高级语言的一次加法，也可能对应一条或多条机器操作，取决于数据类型、宽度、目标硬件与优化。

| 实现方式 | 过程 | 注意点 |
|---|---|---|
| AOT | 运行前生成目标机器码 | 输出通常依赖目标平台 |
| 解释执行 | 解释器执行某种程序表示 | 不一定逐行读源码，也可执行字节码 |
| JIT | 运行时把部分程序转换为机器码 | 可能有预热和编译开销 |
| 混合实现 | 多种方式配合 | 不能只凭语言名称判断 |

例如，C 常见实现会提前编译为本机代码；CPython 通常先把源码编译为字节码，再由解释器执行，调用原生扩展时还会进入本机代码；其他 Python 实现可能采用 JIT。这里说明的是具体实现方式，不是给语言贴永久标签。

## 17. 把整条知识链用在 AI 嵌入式与机器人上

以下为**补充应用分析**。以“摄像头识别目标，再控制电机跟踪”为例：

```text
摄像头数据 → 感知程序 / AI 模型 → 目标位置与控制决策
→ 控制算法与设备驱动 → PWM / 电机控制器 → 电机运动
                                      ↑
                               编码器或其他反馈
```

| 环节 | 与本课程的联系 | 需要额外关注 |
|---|---|---|
| 图像和传感器数据 | 二进制表示、内存与输入输出 | 数据格式、传输量和缓冲 |
| 算法执行 | 指令、ALU、寄存器与控制 | 计算量、并行性和精度 |
| 高性能处理器 | 流水线、缓存、多核与专用加速 | 吞吐量、散热和功耗 |
| 电机周期控制 | 时钟、中断和设备寄存器 | 延迟、抖动、截止时间 |
| 软件开发 | 语言、工具链、库与驱动 | 目标平台、ABI 和调试 |
| 系统部署 | 固件启动与更新、通信接口 | 权限、完整性和失效处理 |

常见任务分配是：较强处理器或 SoC 运行图像处理、AI 推理和规划；MCU 或专门控制单元处理严格周期的采样、PWM 和反馈控制；通信接口交换目标、状态和故障信息。具体划分要依据任务的数据量、计算量、响应时间和功耗预算。

## 18. 一页复习表

| 知识点 | 应能说清的一句话 |
|---|---|
| 漏洞 | 可被利用来违反安全目标的弱点，不必来自恶意 |
| 后门 | 有意绕过正常安全控制的访问或控制机制 |
| 信任链 | 每层依赖受信任的启动、代码和权限机制 |
| 芯片制造 | 器件与多层互连由多轮加工形成，再测试与封装 |
| 光刻 | 形成用于选择加工区域的图形，需要配合其他工序 |
| 工艺节点 | 工艺代际名称，不能代表所有物理尺寸或整机性能 |
| 流水线 | 多条指令同时处于不同阶段，提高连续处理能力 |
| 延迟与吞吐量 | 一条做多久，与一段时间完成多少条，是两种指标 |
| 冒险 | 资源、数据或控制约束使重叠执行需要等待或恢复 |
| 缓存 | 利用局部性减少平均等待，未命中会增加延迟 |
| 超标量 / 乱序 / 多核 | 单周期发射能力 / 核内安排 / 多个执行核心 |
| ISA / 微架构 | 程序可见规则 / 芯片内部实现 |
| ABI | 二进制组件协作所需的调用与数据约定 |
| 编译 / 解释 / JIT | 程序实现的执行方式，可以组合 |
| 实时性 | 在截止时间内完成，不由平均速度单独决定 |

## 19. 自测与参考答案

先遮住答案，再用自己的话回答。

1. 一个远程维护接口就一定是后门吗？
2. 固件验签成功，是否能证明固件没有漏洞？
3. 光刻完成后，是否已经得到整颗可运行的 CPU？
4. 五级流水线稳定后每周期完成一条，是否表示每条延迟只有一个周期？
5. 五级理想流水线执行 6 条指令需要几个周期？相应串行模型需要几个？
6. 为什么 LOAD 后立即使用结果可能仍需要停顿？
7. 分支预测错误时，为什么需要撤销错误路径的工作？
8. 超标量、乱序执行和多核之间有什么区别？
9. 4 核 CPU 是否一定能让同一个程序快 4 倍？
10. 两颗 CPU 的 ISA 相同，所有二进制程序是否都能直接通用？
11. 裸机程序是否一定需要通用操作系统？
12. 为什么机器人中平均性能更强的处理器未必更适合严格周期控制？

### 参考答案

1. 不一定。要看接口是否公开、经过认证和授权，以及是否绕过正常安全控制。
2. 不能。验签支持来源与完整性判断，不证明代码逻辑不存在弱点；防回滚还需要相应机制。
3. 没有。光刻形成加工图形，还要与材料加工、多轮对准、测试和封装等工序配合。
4. 不是。每条仍经过五级；完成频率提高来自多条指令重叠。
5. 流水线为 5+6−1=10 周期；串行是 5×6=30 周期，前提是阶段等长、无开销和停顿。
6. 加载数据可能到 MEM 末尾才可用，而后续执行原本更早需要；旁路不能传递尚未形成的结果。
7. 预测路径可能不符合真实程序流程，不能让其结果错误地成为正式程序状态。
8. 超标量关注一个周期的多指令发射能力；乱序关注内部执行安排；多核提供多个执行核心。
9. 不一定。串行部分、同步通信和共享资源都会限制收益。
10. 不一定。还要满足扩展指令、ABI、操作系统、文件格式和库等条件。
11. 不需要。固件可以在合适的启动和硬件环境中直接运行，也可以按需求使用运行时或 RTOS。
12. 缓存、推测、中断和调度可能带来延迟波动；严格周期控制关注截止时间和可预测性。

### 计算与判断练习

**A. 四级流水线执行 10 条独立指令。**每级一个周期、无停顿：第一条在第 4 周期末完成，最后一条在第 4+10−1=13 周期末完成；相应串行模型需要 40 周期，理想加速比约为 3.08。少量指令仍要承担填充和排空影响。

**B. 机器人每 1 ms 必须更新一次控制输出。**多数更新用时 0.2 ms，偶尔用时 1.4 ms。不能仅凭平均值断言达标：1.4 ms 已超过截止时间，应分析长延迟来源、最坏情况与容许的超时行为。

## 20. 课程对应、资料来源与八篇导航

### 本篇与目录的对应

| 本篇章节 | 课程主题 | 内容来源边界 |
|---|---|---|
| 3—4 | 76_后门和漏洞.mp4 | 标题已核对；定义、案例与保护措施为补充理解 |
| 5—7 | 77_光刻工艺.mp4 | 标题已核对；制造流程和工艺说明为补充理解 |
| 8—12 | 78_流水线技术.mp4 | 标题已核对；五级模型、公式与冒险分析为补充理解 |
| 13—14 | 79_高级CPU技术.mp4 | 标题已核对；缓存、并行及性能分析为通用选讲 |
| 15 | 80_软硬件生态系统.mp4 | 标题已核对；分层和兼容性说明为补充理解 |
| 16 | 81_编程语言发展.mp4 | 标题已核对；语言层次及执行方式为补充理解 |
| 17—19 | 主题综合应用与复习 | 自编应用分析、练习与答案 |

资料来源：[《自己动手做台计算机》课程网盘](https://pan.baidu.com/s/1RLiC8gC1sXTu7EgkQs9oHw?pwd=6633)，提取码：`6633`。目录为“1、自己动手做计算机”；配套讲义为“ppt/自己动手做台计算机.pdf”，共 101 页。

本篇没有为视频 76—81 指定未经核验的讲义页码，没有新增课程电路编码，也没有声称这些补充知识都在视频中出现。

### 八篇笔记导航

| 编号 | 笔记 | 主要问题 |
|---|---|---|
| 01 | [计算机基础与数据表示](01-计算机基础与数据表示.ipynb) | 计算机怎样表示数据？ |
| 02 | [逻辑门、Digital仿真与PCB制作](02-逻辑门、Digital仿真与PCB制作.ipynb) | 逻辑怎样变成可连接、可验证的电路？ |
| 03 | [ALU、加法器与组合逻辑电路](03-ALU、加法器与组合逻辑电路.ipynb) | 电路怎样计算？ |
| 04 | [锁存器、寄存器与内存](04-锁存器、寄存器与内存.ipynb) | 数据怎样被保存？ |
| 05 | [CPU搭建与内部数据通路](05-CPU搭建与内部数据通路.ipynb) | CPU 内部怎样搬运数据？ |
| 06 | [指令集、汇编与手动执行程序](06-指令集、汇编与手动执行程序.ipynb) | 指令怎样规定程序动作？ |
| 07 | [控制单元、自动执行与完整计算机系统](07-控制单元、自动执行与完整计算机系统.ipynb) | 程序怎样被自动执行，系统怎样连接？ |
| 08 | 芯片制造、CPU进阶与软硬件生态 | 真实系统怎样制造、加速、协作并建立信任？ |

建议最终复习沿两条线检查理解：

```text
物理与硬件线：器件制造 → 逻辑门 → ALU与寄存器 → CPU数据通路
            → 控制单元 → 内存与输入输出 → 性能与时序
程序与系统线：数据表示 → 机器指令 → 汇编与高级语言 → 工具链
            → 固件或操作系统 → 应用与控制任务 → 权限与安全
```

能够说明“某条指令读取哪些数据、经过哪些硬件、更新什么状态”，并把它放回制造、工具链和应用环境中，就建立起继续学习嵌入式系统的底层框架。